# Studi Komparatif Model Generatif: Vanilla AE, Sparse AE, VAE, GAN, DCGAN, DDPM
**Deep Learning (D6421) Sesi 07 - Tugas Take-Home** · Dataset: **Fashion-MNIST**

Langkah:
1. `Runtime -> Change runtime type -> T4 GPU`
2. (Opsional) set `USE_DRIVE = True` agar checkpoint & figur tersimpan di Google Drive (aman jika runtime terputus).
3. `Runtime -> Run all`. Estimasi total waktu sekitar 60-90 menit di T4.

Urutan notebook: setup -> **Bagian 2** (MLP Classifier, Tahap 1-5) -> Vanilla AE / Sparse AE / VAE -> Vanilla GAN / DCGAN -> DDPM
-> tabel metrik (Bagian IV) -> lima figur wajib (Bagian V) -> unduh hasil.

In [ ]:
!nvidia-smi -L || echo "GPU tidak terdeteksi - aktifkan T4 GPU di Runtime settings"
!pip -q install "torchmetrics[image]" torch-fidelity tabulate

In [ ]:
import os
USE_DRIVE = False   # True -> simpan outputs ke Google Drive
QUICK = False       # True -> smoke test cepat (subset kecil, 1 epoch)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['OUT_DIR'] = '/content/drive/MyDrive/generative_models_outputs'
else:
    os.environ['OUT_DIR'] = '/content/outputs'
os.environ['DATA_DIR'] = '/content/data'
os.environ['QUICK'] = '1' if QUICK else '0'
os.makedirs('/content/src', exist_ok=True)
%cd /content

## Kode modular
Sel-sel berikut menulis paket `src/` (sama persis dengan isi repositori GitHub).

In [ ]:
%%writefile src/config.py
"""Konfigurasi global eksperimen: seed, device, hyperparameter, dan path output.

Set environment variable QUICK=1 untuk smoke test cepat (subset data kecil, 1 epoch,
T DDPM kecil). Untuk hasil laporan, jalankan tanpa QUICK di GPU (Colab T4).
"""
import os
import random

import numpy as np
import torch

SEED = 42
QUICK = os.environ.get("QUICK", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = os.environ.get("DATA_DIR", "./data")
OUT_DIR = os.environ.get("OUT_DIR", "./outputs")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
FIG_DIR = os.path.join(OUT_DIR, "figures")

# ---------------- Data ----------------
DATASET = "FashionMNIST"
BATCH_SIZE = 128
SPLIT = (0.8, 0.1, 0.1)                 # train / val / test dari 70.000 citra
QUICK_SUBSET = 2048                      # jumlah citra train saat QUICK=1

# ---------------- Arsitektur ----------------
LATENT_AE = 64                           # dim laten Vanilla AE, Sparse AE, VAE, MLP z
SPARSE_HIDDEN = 256                      # lapisan tersembunyi yang diberi penalti L1
SPARSE_LAMBDA = 1e-4                     # bobot penalti lambda * ||h||_1
Z_GAN = 100                              # dim prior noise Vanilla GAN & DCGAN
DDPM_T = 50 if QUICK else 1000           # jumlah langkah difusi
DDPM_BETA = (1e-4, 0.02)                 # linear beta schedule (Ho et al., 2020)
DDPM_BASE_CH = 64

# ---------------- Training ----------------
EPOCHS = dict(mlp=20, frozen_dec=20, ae=20, sae=20, vae=30, gan=50, dcgan=25, ddpm=30)
if QUICK:
    EPOCHS = {k: 1 for k in EPOCHS}
LR = dict(mlp=1e-3, frozen_dec=1e-3, ae=1e-3, sae=1e-3, vae=1e-3, gan=2e-4, dcgan=2e-4, ddpm=2e-4)

# ---------------- Evaluasi ----------------
FID_N = 200 if QUICK else 2000           # jumlah sampel sintetis & real untuk FID
GEN_BATCH = 250                          # ukuran potongan saat generate sampel (hemat VRAM, aman utk GPU 6GB)
FID_BATCH = 100                          # ukuran potongan saat ekstraksi fitur InceptionV3 (299x299)
LATENCY_BATCH = 64
INTERP_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
INTERP_CLASSES = (0, 9)                  # T-shirt/top  vs  Ankle boot (kontras)

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def set_seed(seed: int = SEED) -> None:
    """Menetapkan seed agar seluruh eksperimen reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_dirs() -> None:
    for d in (OUT_DIR, CKPT_DIR, FIG_DIR):
        os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/data.py
"""Pemuatan dataset Fashion-MNIST dengan split 80/10/10 yang seragam untuk semua model.

Train (60k) dan test (10k) bawaan torchvision digabung menjadi 70.000 citra, lalu dibagi
secara acak dengan generator ber-seed tetap -> 56.000 / 7.000 / 7.000.
Citra disimpan dalam rentang [0, 1]. Model GAN/DCGAN/DDPM mengubahnya ke [-1, 1] sendiri.
"""
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

from . import config as C


def _load_tensors():
    tr = datasets.FashionMNIST(C.DATA_DIR, train=True, download=True)
    te = datasets.FashionMNIST(C.DATA_DIR, train=False, download=True)
    x = torch.cat([tr.data, te.data]).float().div(255.0).unsqueeze(1)   # (70000,1,28,28)
    y = torch.cat([tr.targets, te.targets]).long()
    return x, y


def get_splits():
    """Mengembalikan dict berisi tensor (x, y) untuk train/val/test."""
    x, y = _load_tensors()
    n = len(x)
    n_tr, n_va = int(C.SPLIT[0] * n), int(C.SPLIT[1] * n)
    g = torch.Generator().manual_seed(C.SEED)
    perm = torch.randperm(n, generator=g)
    idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    if C.QUICK:
        idx = {"train": idx["train"][:C.QUICK_SUBSET],
               "val": idx["val"][:C.QUICK_SUBSET // 4],
               "test": idx["test"][:max(C.QUICK_SUBSET // 4, C.FID_N)]}
    return {k: (x[v], y[v]) for k, v in idx.items()}


def get_loaders(splits, batch_size: int = C.BATCH_SIZE):
    g = torch.Generator().manual_seed(C.SEED)
    loaders = {}
    for name, (x, y) in splits.items():
        loaders[name] = DataLoader(TensorDataset(x, y), batch_size=batch_size,
                                   shuffle=(name == "train"), generator=g if name == "train" else None,
                                   drop_last=False)
    return loaders

In [ ]:
%%writefile src/models.py
"""Definisi arsitektur: Vanilla AE, Sparse AE, VAE, Vanilla GAN (MLP), DCGAN, DDPM, MLP Classifier.

Setiap model keluarga autoencoder menyediakan:
  - loss(x)         -> (loss_scalar, dict_log)      dipakai oleh training loop generik
  - reconstruct(x)  -> citra rekonstruksi [0,1]
  - encode(x) / decode(z)
"""
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from . import config as C


# =====================================================================================
# 1) Vanilla Autoencoder  : 784 -> 256 -> 64 -> 256 -> 784
#    L = || x - x_hat ||^2  (MSE)
# =====================================================================================
class VanillaAE(nn.Module):
    def __init__(self, latent: int = C.LATENT_AE, hidden: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(),
                                     nn.Linear(hidden, latent))
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

    def reconstruct(self, x):
        return self.forward(x)[0]

    def loss(self, x):
        x_hat, _ = self.forward(x)
        mse = F.mse_loss(x_hat, x)
        return mse, {"recon": mse.item()}


# =====================================================================================
# 2) Sparse Autoencoder : 784 -> 256 (sparse, penalti L1) -> 64 -> 256 -> 784
#    L = || x - x_hat ||^2 + lambda * || h ||_1
# =====================================================================================
class SparseAE(nn.Module):
    def __init__(self, latent: int = C.LATENT_AE, hidden: int = C.SPARSE_HIDDEN,
                 lam: float = C.SPARSE_LAMBDA):
        super().__init__()
        self.lam = lam
        self.enc1 = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU())
        self.enc2 = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def hidden(self, x):
        return self.enc1(x)

    def encode(self, x):
        return self.enc2(self.enc1(x))

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        h = self.enc1(x)
        z = self.enc2(h)
        return self.decode(z), z, h

    def reconstruct(self, x):
        return self.forward(x)[0]

    def loss(self, x):
        x_hat, _, h = self.forward(x)
        mse = F.mse_loss(x_hat, x)
        l1 = h.abs().sum(dim=1).mean()
        return mse + self.lam * l1, {"recon": mse.item(), "l1": l1.item()}


# =====================================================================================
# 3) Variational Autoencoder : 784 -> 256 -> (mu, logvar) in R^64 -> 256 -> 784
#    L = E_q[-log p(x|z)] (BCE) + KL( q(z|x) || N(0, I) )
# =====================================================================================
class VAE(nn.Module):
    def __init__(self, latent: int = C.LATENT_AE, hidden: int = 256):
        super().__init__()
        self.latent = latent
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU())
        self.fc_mu = nn.Linear(hidden, latent)
        self.fc_logvar = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def encode_dist(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def encode(self, x):          # dipakai untuk interpolasi -> mean posterior
        return self.encode_dist(x)[0]

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode_dist(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)   # reparameterization trick
        return self.decode(z), mu, logvar

    def reconstruct(self, x):
        return self.decode(self.encode(x))

    def sample(self, n, device=C.DEVICE):
        return self.decode(torch.randn(n, self.latent, device=device))

    def loss(self, x):
        x_hat, mu, logvar = self.forward(x)
        b = x.size(0)
        bce = F.binary_cross_entropy(x_hat, x, reduction="sum") / b
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / b
        return bce + kl, {"recon": bce.item(), "kl": kl.item(),
                          "mse": F.mse_loss(x_hat, x).item()}


# =====================================================================================
# 4) Vanilla GAN (fully connected)
#    min_G max_D  E[log D(x)] + E[log(1 - D(G(z)))]   (G memakai non-saturating loss)
# =====================================================================================
class MLPGenerator(nn.Module):
    def __init__(self, z_dim: int = C.Z_GAN):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, 784), nn.Tanh())

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class MLPDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1))        # logit

    def forward(self, x):
        return self.net(x).view(-1)


# =====================================================================================
# 5) DCGAN (Radford et al., 2016) disesuaikan untuk 28x28
#    G: z(100) -> ConvT 7x7 -> 14x14 -> 28x28, BatchNorm + ReLU, output Tanh
#    D: Conv stride-2 28 -> 14 -> 7 -> 1, BatchNorm + LeakyReLU, tanpa pooling / FC
# =====================================================================================
class DCGenerator(nn.Module):
    def __init__(self, z_dim: int = C.Z_GAN, ngf: int = 64):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ngf * 4, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, 1, 4, 2, 1, bias=False), nn.Tanh())

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class DCDiscriminator(nn.Module):
    def __init__(self, ndf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, 1, 7, 1, 0, bias=False))

    def forward(self, x):
        return self.net(x).view(-1)


def dcgan_init(m):
    """Inisialisasi bobot N(0, 0.02) sesuai paper DCGAN."""
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


# =====================================================================================
# 6) DDPM (Ho et al., 2020) dengan U-Net kecil
#    q(x_t|x_0) = N( sqrt(abar_t) x_0, (1-abar_t) I )
#    L_simple = E_{t,x0,eps} || eps - eps_theta(x_t, t) ||^2
# =====================================================================================
class SinusoidalEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t.float()[:, None] * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, 1, 1)
        self.t = nn.Linear(tdim, cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, 1, 1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.t(temb)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)


class UNet(nn.Module):
    """U-Net 28 -> 14 -> 7 -> 14 -> 28 dengan skip connection dan time embedding."""

    def __init__(self, ch: int = C.DDPM_BASE_CH, tdim: int = 256):
        super().__init__()
        self.temb = nn.Sequential(SinusoidalEmb(ch), nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inc = nn.Conv2d(1, ch, 3, 1, 1)
        self.d1 = ResBlock(ch, ch, tdim)
        self.ds1 = nn.Conv2d(ch, ch, 4, 2, 1)                      # 28 -> 14
        self.d2 = ResBlock(ch, ch * 2, tdim)
        self.ds2 = nn.Conv2d(ch * 2, ch * 2, 4, 2, 1)              # 14 -> 7
        self.m1 = ResBlock(ch * 2, ch * 4, tdim)
        self.m2 = ResBlock(ch * 4, ch * 4, tdim)
        self.us2 = nn.ConvTranspose2d(ch * 4, ch * 2, 4, 2, 1)     # 7 -> 14
        self.u2 = ResBlock(ch * 4, ch * 2, tdim)
        self.us1 = nn.ConvTranspose2d(ch * 2, ch, 4, 2, 1)         # 14 -> 28
        self.u1 = ResBlock(ch * 2, ch, tdim)
        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, 1, 3, 1, 1))

    def forward(self, x, t):
        te = self.temb(t)
        h1 = self.d1(self.inc(x), te)                 # 28x28
        h2 = self.d2(self.ds1(h1), te)                # 14x14
        m = self.m2(self.m1(self.ds2(h2), te), te)    # 7x7
        u = self.u2(torch.cat([self.us2(m), h2], 1), te)
        u = self.u1(torch.cat([self.us1(u), h1], 1), te)
        return self.out(u)


class DDPM(nn.Module):
    def __init__(self, T: int = C.DDPM_T, beta=C.DDPM_BETA, ema_decay: float = 0.999):
        super().__init__()
        self.T = T
        self.net = UNet()
        self.ema = copy.deepcopy(self.net).requires_grad_(False)   # bobot EMA untuk sampling
        self.ema_decay = ema_decay
        betas = torch.linspace(beta[0], beta[1], T)
        alphas = 1.0 - betas
        abar = torch.cumprod(alphas, 0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("abar", abar)

    @torch.no_grad()
    def update_ema(self):
        for pe, p in zip(self.ema.parameters(), self.net.parameters()):
            pe.mul_(self.ema_decay).add_(p.detach(), alpha=1 - self.ema_decay)
        for be, b in zip(self.ema.buffers(), self.net.buffers()):
            be.copy_(b)

    def q_sample(self, x0, t, noise):
        ab = self.abar[t].view(-1, 1, 1, 1)
        return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

    def loss(self, x):
        x0 = x * 2 - 1                                   # [0,1] -> [-1,1]
        t = torch.randint(0, self.T, (x0.size(0),), device=x0.device)
        noise = torch.randn_like(x0)
        pred = self.net(self.q_sample(x0, t, noise), t)
        mse = F.mse_loss(pred, noise)
        return mse, {"noise_mse": mse.item()}

    @torch.no_grad()
    def sample(self, n, device=C.DEVICE, traj_every: int = 0):
        """Reverse process x_T ~ N(0,I) -> x_0. Mengembalikan citra [0,1] (dan trajektori)."""
        net = self.ema
        x = torch.randn(n, 1, 28, 28, device=device)
        traj = []
        for t in reversed(range(self.T)):
            tt = torch.full((n,), t, device=device, dtype=torch.long)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                eps = net(x, tt)
            eps = eps.float()
            a, ab, b = self.alphas[t], self.abar[t], self.betas[t]
            mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
            x = mean + b.sqrt() * torch.randn_like(x) if t > 0 else mean
            if traj_every and (t % traj_every == 0 or t == self.T - 1):
                traj.append((t, ((x.clamp(-1, 1) + 1) / 2).cpu()))
        out = (x.clamp(-1, 1) + 1) / 2
        return (out, traj) if traj_every else out


# =====================================================================================
# 7) MLP Classifier pendamping : 784 -> 256 -> 64 -> 10   (struktur identik encoder AE)
#    L = CrossEntropy(y, softmax(f(x)))  -> memodelkan P(y|x)
# =====================================================================================
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, C.LATENT_AE)
        self.fc3 = nn.Linear(C.LATENT_AE, 10)

    def features(self, x):
        """z_MLP in R^64: representasi tepat sebelum lapisan klasifikasi."""
        h = F.relu(self.fc1(x.flatten(1)))
        return F.relu(self.fc2(h))

    def forward(self, x):
        return self.fc3(self.features(x))

    def loss(self, x, y):
        logits = self.forward(x)
        ce = F.cross_entropy(logits, y)
        return ce, {"ce": ce.item(), "acc": (logits.argmax(1) == y).float().mean().item()}


class FrozenLatentDecoder(nn.Module):
    """Decoder MLP 64 -> 256 -> 784 untuk Tahap 4 (dilatih di atas z_MLP yang dibekukan)."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(C.LATENT_AE, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

In [ ]:
%%writefile src/train.py
"""Training loop: generik (AE / Sparse AE / VAE / DDPM / MLP / Frozen decoder) dan adversarial (GAN / DCGAN).

Setiap fungsi mengembalikan `history` (dict of lists) yang berisi loss per-epoch,
loss per-iterasi (untuk analisis osilasi), dan waktu training per epoch.
Checkpoint + history disimpan sehingga training bisa dilewati bila sudah pernah dijalankan
(berguna saat runtime Colab terputus dan OUT_DIR berada di Google Drive).
"""
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn

from . import config as C


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def _paths(name):
    return os.path.join(C.CKPT_DIR, f"{name}.pt"), os.path.join(C.CKPT_DIR, f"{name}_history.json")


def try_load(name, modules: dict):
    ck, hp = _paths(name)
    if os.path.exists(ck) and os.path.exists(hp):
        state = torch.load(ck, map_location=C.DEVICE)
        for k, m in modules.items():
            m.load_state_dict(state[k])
        with open(hp) as f:
            print(f"[{name}] checkpoint ditemukan -> training dilewati")
            return json.load(f)
    return None


def save(name, modules: dict, history):
    ck, hp = _paths(name)
    torch.save({k: m.state_dict() for k, m in modules.items()}, ck)
    with open(hp, "w") as f:
        json.dump(history, f)


def train_generic(name, model, loaders, epochs, lr, loss_fn=None, ema=False, amp=False):
    """loss_fn(model, x, y) -> (loss, logs). Default: model.loss(x)."""
    model.to(C.DEVICE)
    hist = try_load(name, {"model": model})
    if hist is not None:
        return hist
    loss_fn = loss_fn or (lambda m, x, y: m.loss(x))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    use_amp = amp and C.DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist = {"train_loss": [], "val_loss": [], "epoch_time": [], "iter_loss": [], "logs": []}
    for ep in range(epochs):
        model.train()
        _sync()
        t0 = time.time()
        losses, logs_acc = [], {}
        for x, y in loaders["train"]:
            x, y = x.to(C.DEVICE, non_blocking=True), y.to(C.DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss, logs = loss_fn(model, x, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if ema:
                model.update_ema()
            losses.append(loss.item())
            for k, v in logs.items():
                logs_acc.setdefault(k, []).append(v)
        _sync()
        hist["epoch_time"].append(time.time() - t0)
        hist["train_loss"].append(float(np.mean(losses)))
        hist["iter_loss"].extend(losses)
        hist["logs"].append({k: float(np.mean(v)) for k, v in logs_acc.items()})
        hist["val_loss"].append(evaluate_loss(model, loaders["val"], loss_fn))
        print(f"[{name}] epoch {ep + 1:3d}/{epochs}  train={hist['train_loss'][-1]:.4f}  "
              f"val={hist['val_loss'][-1]:.4f}  {hist['logs'][-1]}  ({hist['epoch_time'][-1]:.1f}s)")
    save(name, {"model": model}, hist)
    return hist


@torch.no_grad()
def evaluate_loss(model, loader, loss_fn):
    model.eval()
    tot, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(C.DEVICE), y.to(C.DEVICE)
        loss, _ = loss_fn(model, x, y)
        tot += loss.item() * x.size(0)
        n += x.size(0)
    return tot / n


def _grad_norm(module):
    g = [p.grad.detach().norm() for p in module.parameters() if p.grad is not None]
    return torch.stack(g).norm().item() if g else 0.0


def train_gan(name, G, D, loaders, epochs, lr, z_dim=C.Z_GAN):
    """Minimax GAN dengan BCE; generator memakai non-saturating loss  -log D(G(z))."""
    G.to(C.DEVICE)
    D.to(C.DEVICE)
    hist = try_load(name, {"G": G, "D": D})
    if hist is not None:
        return hist
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    hist = {k: [] for k in ["g_loss", "d_loss", "d_real", "d_fake", "g_grad", "epoch_time",
                            "iter_g", "iter_d"]}
    for ep in range(epochs):
        G.train()
        D.train()
        _sync()
        t0 = time.time()
        acc = {k: [] for k in ["g", "d", "dr", "df", "gg"]}
        for x, _ in loaders["train"]:
            x = x.to(C.DEVICE) * 2 - 1                    # [0,1] -> [-1,1]
            b = x.size(0)
            ones = torch.ones(b, device=C.DEVICE)
            zeros = torch.zeros(b, device=C.DEVICE)
            fake = G(torch.randn(b, z_dim, device=C.DEVICE))

            # ---- update Discriminator: max log D(x) + log(1 - D(G(z)))
            out_r, out_f = D(x), D(fake.detach())
            d_loss = bce(out_r, ones) + bce(out_f, zeros)
            optD.zero_grad(set_to_none=True)
            d_loss.backward()
            optD.step()

            # ---- update Generator: min -log D(G(z))
            out_g = D(fake)
            g_loss = bce(out_g, ones)
            optG.zero_grad(set_to_none=True)
            g_loss.backward()
            acc["gg"].append(_grad_norm(G))
            optG.step()

            acc["g"].append(g_loss.item())
            acc["d"].append(d_loss.item())
            acc["dr"].append(torch.sigmoid(out_r).mean().item())
            acc["df"].append(torch.sigmoid(out_f).mean().item())
        _sync()
        hist["epoch_time"].append(time.time() - t0)
        for hk, ak in [("g_loss", "g"), ("d_loss", "d"), ("d_real", "dr"), ("d_fake", "df"), ("g_grad", "gg")]:
            hist[hk].append(float(np.mean(acc[ak])))
        hist["iter_g"].extend(acc["g"])
        hist["iter_d"].extend(acc["d"])
        print(f"[{name}] epoch {ep + 1:3d}/{epochs}  G={hist['g_loss'][-1]:.3f}  D={hist['d_loss'][-1]:.3f}  "
              f"D(x)={hist['d_real'][-1]:.2f}  D(G(z))={hist['d_fake'][-1]:.2f}  "
              f"|gradG|={hist['g_grad'][-1]:.3f}  ({hist['epoch_time'][-1]:.1f}s)")
    save(name, {"G": G, "D": D}, hist)
    return hist

In [ ]:
%%writefile src/metrics.py
"""Metrik evaluasi: MSE, PSNR, SSIM (rekonstruksi), FID (sampel sintetis), latency, mode coverage."""
import time

import numpy as np
import torch
from torchmetrics.functional.image import structural_similarity_index_measure as ssim_fn

from . import config as C


@torch.no_grad()
def recon_metrics(recon_fn, loader):
    """recon_fn(x)->x_hat di [0,1]. Mengembalikan rata-rata per-citra MSE, PSNR(dB), SSIM pada loader."""
    mse_all, psnr_all, ssim_sum, n = [], [], 0.0, 0
    for x, _ in loader:
        x = x.to(C.DEVICE)
        x_hat = recon_fn(x).clamp(0, 1)
        mse = ((x_hat - x) ** 2).flatten(1).mean(1)
        mse_all.append(mse.cpu())
        psnr_all.append((10 * torch.log10(1.0 / mse.clamp_min(1e-10))).cpu())
        ssim_sum += ssim_fn(x_hat, x, data_range=1.0).item() * x.size(0)
        n += x.size(0)
    return {"mse": torch.cat(mse_all).mean().item(),
            "psnr": torch.cat(psnr_all).mean().item(),
            "ssim": ssim_sum / n}


@torch.no_grad()
def generate(sample_fn, n, batch=C.GEN_BATCH):
    """sample_fn(k)->k citra [0,1]. Digenerate bertahap agar hemat memori."""
    out = []
    while sum(o.size(0) for o in out) < n:
        k = min(batch, n - sum(o.size(0) for o in out))
        out.append(sample_fn(k).clamp(0, 1).cpu())
    return torch.cat(out)


@torch.no_grad()
def fid_score(real, fake, batch=C.FID_BATCH):
    """FID (InceptionV3 pool3, 2048-d). Citra grayscale direplikasi ke 3 kanal; resize ke 299 dilakukan internal."""
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(C.DEVICE)
    for imgs, is_real in ((real, True), (fake, False)):
        for i in range(0, len(imgs), batch):
            fid.update(imgs[i:i + batch].to(C.DEVICE).repeat(1, 3, 1, 1), real=is_real)
    return fid.compute().item()


@torch.no_grad()
def latency_ms(fn, repeats=10, warmup=2):
    """Rata-rata waktu (ms) satu pemanggilan fn() yang menghasilkan satu batch (64 citra)."""
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))


@torch.no_grad()
def mode_coverage(classifier, imgs):
    """Distribusi kelas (menurut MLP Classifier) dari sampel sintetis.
    Mengembalikan entropi ternormalisasi (1.0 = 10 kelas merata; rendah = indikasi mode collapse)
    dan histogram kelas."""
    classifier.eval()
    preds = []
    for i in range(0, len(imgs), 500):
        preds.append(classifier(imgs[i:i + 500].to(C.DEVICE)).argmax(1).cpu())
    hist = torch.bincount(torch.cat(preds), minlength=10).float()
    p = hist / hist.sum()
    ent = max(0.0, -(p[p > 0] * p[p > 0].log()).sum().item() / np.log(10))
    return ent, hist.long().tolist()


def stability_cv(iter_losses, tail_frac=0.5):
    """Koefisien variasi loss per-iterasi pada separuh akhir training (indikator osilasi)."""
    a = np.asarray(iter_losses[int(len(iter_losses) * (1 - tail_frac)):])
    return float(a.std() / (abs(a.mean()) + 1e-8))

In [ ]:
%%writefile src/mlp_limits.py
"""Bagian 2: pembuktian empiris bahwa bobot & fitur laten MLP Classifier tidak bisa dipakai untuk generasi citra.

Tahap 1 : latih MLP 784-256-64-10 (CE) -> akurasi test >= 88%          (experiment.py)
Tahap 2 : proyeksi balik transpos dari one-hot kelas   x = s(W1^T r(W2^T r(W3^T y_c)))
Tahap 3 : proyeksi balik transpos dari z_MLP citra uji x = s(W1^T r(W2^T z_MLP))
Tahap 4 : frozen latent decoder 64-256-784 (MSE) vs Vanilla AE z in R^64  (experiment.py)
Tahap 5 : sampling z ~ N(mu_z, diag(sigma_z^2)) -> decoder Tahap 4
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

from . import config as C


class FrozenPipe(nn.Module):
    """Classifier beku (requires_grad=False) + decoder yang dilatih di atas z_MLP."""

    def __init__(self, classifier, decoder):
        super().__init__()
        self.classifier = classifier.requires_grad_(False)
        self.decoder = decoder

    def train(self, mode=True):
        super().train(mode)
        self.classifier.eval()          # classifier selalu mode eval
        return self

    def reconstruct(self, x):
        with torch.no_grad():
            z = self.classifier.features(x)
        return self.decoder(z)

    def loss(self, x):
        x_hat = self.reconstruct(x)
        mse = F.mse_loss(x_hat, x)
        return mse, {"recon": mse.item()}


def weights(clf):
    return clf.fc1.weight.detach(), clf.fc2.weight.detach(), clf.fc3.weight.detach()   # (256,784),(64,256),(10,64)


@torch.no_grad()
def transpose_from_class(clf):
    """Tahap 2: one-hot y_c (10 kelas) -> ruang piksel via transpos bobot (tanpa bias, tanpa training)."""
    W1, W2, W3 = weights(clf)
    y = torch.eye(10, device=W1.device)
    z = F.relu(y @ W3)          # (10,64)   = ReLU(W3^T y_c)
    h = F.relu(z @ W2)          # (10,256)  = ReLU(W2^T z)
    pre = h @ W1                # (10,784)  = W1^T h
    return torch.sigmoid(pre).view(-1, 1, 28, 28), pre.view(-1, 1, 28, 28)


@torch.no_grad()
def transpose_from_latent(clf, x):
    """Tahap 3: z_MLP citra uji -> ruang piksel via transpos bobot."""
    W1, W2, _ = weights(clf)
    z = clf.features(x)
    pre = F.relu(z @ W2) @ W1
    return torch.sigmoid(pre).view(-1, 1, 28, 28), pre.view(-1, 1, 28, 28)


@torch.no_grad()
def pinv_from_latent(clf, x):
    """Pembanding: invers semu Moore-Penrose (W^+) termasuk bias -- tetap gagal karena
    dimensi direduksi (784 -> 64) dan ReLU sudah membuang nilai negatif."""
    W1, W2, _ = weights(clf)
    b1, b2 = clf.fc1.bias.detach(), clf.fc2.bias.detach()
    z = clf.features(x)
    h = (z - b2) @ torch.linalg.pinv(W2).T
    xr = (h - b1) @ torch.linalg.pinv(W1).T
    return xr.clamp(0, 1).view(-1, 1, 28, 28)


@torch.no_grad()
def weight_diagnostics(clf, loader):
    """Bukti numerik: ||W^T W - I||_F, ||W^T - W^+||_F, rank, dan fraksi aktivasi ReLU = 0."""
    W1, W2, W3 = weights(clf)
    out = {}
    for name, W in (("W1", W1), ("W2", W2), ("W3", W3)):
        n_in = W.shape[1]
        WtW = W.T @ W
        out[name] = {
            "shape": list(W.shape),
            "rank": int(torch.linalg.matrix_rank(W).item()),
            "input_dim": n_in,
            "||W^T W - I||_F": float(torch.linalg.norm(WtW - torch.eye(n_in, device=W.device))),
            "||W^T - W^+||_F": float(torch.linalg.norm(W.T - torch.linalg.pinv(W))),
        }
    dead1, dead2, n = 0.0, 0.0, 0
    for x, _ in loader:
        x = x.to(C.DEVICE)
        h1 = F.relu(clf.fc1(x.flatten(1)))
        h2 = F.relu(clf.fc2(h1))
        dead1 += (h1 == 0).float().mean().item() * x.size(0)
        dead2 += (h2 == 0).float().mean().item() * x.size(0)
        n += x.size(0)
    out["relu_zero_frac_layer1"] = dead1 / n
    out["relu_zero_frac_layer2"] = dead2 / n
    return out


@torch.no_grad()
def latent_stats(clf, loader):
    zs, ys = [], []
    for x, y in loader:
        zs.append(clf.features(x.to(C.DEVICE)).cpu())
        ys.append(y)
    z, y = torch.cat(zs), torch.cat(ys)
    return z, y, z.mean(0), z.std(0)


@torch.no_grad()
def intra_class_variance_ratio(z, y):
    """Rasio varians intra-kelas / total pada ruang fitur. Nilai kecil = variasi intra-kelas dibuang."""
    total = z.var(0).sum()
    intra = sum(z[y == c].var(0).sum() * (y == c).sum() for c in range(10)) / len(y)
    return float(intra / total)


@torch.no_grad()
def prior_sampling(decoder, mu, sigma, n, generator=None):
    """Tahap 5: z_rand ~ N(mu_z, diag(sigma_z^2)) lalu didekode."""
    eps = torch.randn(n, mu.numel(), generator=generator)
    z = (mu + sigma * eps).to(C.DEVICE)
    return decoder(z)

In [ ]:
%%writefile src/plots.py
"""Pembuatan lima figur wajib (dan figur pendukung) untuk laporan. Semua disimpan PNG 200 dpi."""
import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.utils import make_grid

from . import config as C

matplotlib.rcParams.update({"font.size": 10, "axes.titlesize": 11, "figure.dpi": 100})


def _np(grid):
    return grid.permute(1, 2, 0).cpu().numpy()


def _row(imgs, nrow=None):
    imgs = imgs.detach().float().cpu().clamp(0, 1)
    return _np(make_grid(imgs, nrow=nrow or len(imgs), padding=2, pad_value=1.0))


def _save(fig, name):
    path = os.path.join(C.FIG_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved:", path)
    return path


def stacked_rows(rows, title, fname, label_width=0.18):
    """rows: list of (label, tensor_imgs) atau (label, tensor_imgs, nrow). Satu panel per label."""
    heights = [int(np.ceil(len(r[1]) / (r[2] if len(r) > 2 else len(r[1])))) for r in rows]
    fig, axes = plt.subplots(len(rows), 1, figsize=(12, 1.45 * sum(heights) + 0.6),
                             gridspec_kw={"height_ratios": heights})
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, rows):
        label, imgs = r[0], r[1]
        ax.imshow(_row(imgs, nrow=r[2] if len(r) > 2 else None), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=10)
    fig.suptitle(title, fontsize=13, y=1.0)
    fig.tight_layout()
    return fig, _save(fig, fname)


# ---------------------------------------------------------------- Figur 1
def figure1_reconstruction(x, recons: dict):
    rows = [("Ground Truth", x)] + [(k, v) for k, v in recons.items()]
    return stacked_rows(rows, "Figur 1 - Rekonstruksi Test Set: Vanilla AE vs Sparse AE vs VAE",
                        "fig1_reconstruction.png")


# ---------------------------------------------------------------- Figur 2
def figure2_sampling(samples: dict):
    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4.4))
    for ax, (k, v) in zip(axes, samples.items()):
        ax.imshow(_row(v[:16], nrow=4), cmap="gray")
        ax.set_title(k)
        ax.axis("off")
    fig.suptitle("Figur 2 - Random Generative Sampling (4x4) dari prior N(0, I)", fontsize=13)
    fig.tight_layout()
    return fig, _save(fig, "fig2_random_sampling.png")


# ---------------------------------------------------------------- Figur 3
def _minmax(t):
    lo, hi = t.amin(dim=(1, 2, 3), keepdim=True), t.amax(dim=(1, 2, 3), keepdim=True)
    return (t - lo) / (hi - lo + 1e-8)


def figure3_mlp_limits(x, wt_class, wt_latent, pinv_latent, frozen_rec, ae_rec, prior_samples, vae_samples,
                       wt_class_pre=None, wt_latent_pre=None):
    rows = [
        ("(1) Ground Truth", x),
        ("(2a) Sigmoid(W^T ... y_c)\none-hot kelas 0..9", wt_class),
    ]
    if wt_class_pre is not None:
        rows.append(("(2a') sama, kontras\ndinormalisasi min-max", _minmax(wt_class_pre)))
    rows.append(("(2b) Sigmoid(W^T ... z_MLP)\n(citra uji)", wt_latent))
    if wt_latent_pre is not None:
        rows.append(("(2b') sama, kontras\ndinormalisasi min-max", _minmax(wt_latent_pre)))
    rows += [
        ("(2c) pseudo-inverse W^+\n(pembanding)", pinv_latent),
        ("(3a) Frozen Latent Decoder\n(z_MLP)", frozen_rec),
        ("(3b) Vanilla AE (z_AE)", ae_rec),
        ("(4a) Sampling z~N(mu_z, s_z^2)\nruang laten MLP", prior_samples, 8),
        ("(4b) Pembanding: VAE\nz~N(0, I)", vae_samples, 8),
    ]
    return stacked_rows(rows, "Figur 3 - Bukti Empiris Batasan Bobot & Latent Features MLP Classifier",
                        "fig3_mlp_classifier_limits.png")


# ---------------------------------------------------------------- Figur 4
def figure4_interpolation(interps: dict, alphas):
    fig, axes = plt.subplots(len(interps), 1, figsize=(7.5, 1.45 * len(interps) + 0.9))
    for ax, (k, v) in zip(axes, interps.items()):
        ax.imshow(_row(v), cmap="gray")
        ax.set_yticks([])
        ax.set_ylabel(k, rotation=0, ha="right", va="center")
        ax.set_xticks([(i + 0.5) * 30 for i in range(len(alphas))])
        ax.set_xticklabels([f"a={a:.1f}" for a in alphas], fontsize=8)
    fig.suptitle("Figur 4 - Linear Latent Interpolation  z(a) = (1-a) z_A + a z_B", fontsize=13)
    fig.tight_layout()
    return fig, _save(fig, "fig4_latent_interpolation.png")


# ---------------------------------------------------------------- Figur 5
def figure5_losses_and_ddpm(H: dict, traj):
    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.9])

    def curve(ax, title, series, ylabel="loss", logy=False):
        for label, ys in series:
            ax.plot(np.arange(1, len(ys) + 1), ys, label=label, lw=1.6, marker=".", ms=4)
        ax.set_title(title)
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylabel)
        if logy:
            ax.set_yscale("log")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    curve(fig.add_subplot(gs[0, 0]), "Vanilla AE (MSE)", [("train", H["ae"]["train_loss"]), ("val", H["ae"]["val_loss"])])
    sae_rec = [l["recon"] for l in H["sae"]["logs"]]
    curve(fig.add_subplot(gs[0, 1]), "Sparse AE (MSE + lambda*L1)",
          [("total train", H["sae"]["train_loss"]), ("total val", H["sae"]["val_loss"]), ("MSE train", sae_rec)])
    curve(fig.add_subplot(gs[0, 2]), "VAE (BCE + KL, per citra)",
          [("total train", H["vae"]["train_loss"]), ("total val", H["vae"]["val_loss"]),
           ("KL train", [l["kl"] for l in H["vae"]["logs"]])])
    for j, (key, title) in enumerate([("gan", "Vanilla GAN (MLP)"), ("dcgan", "DCGAN")]):
        ax = fig.add_subplot(gs[1, j])
        it_g, it_d = np.array(H[key]["iter_g"]), np.array(H[key]["iter_d"])
        ipe = max(1, len(it_g) // max(1, len(H[key]["g_loss"])))
        xs = np.arange(len(it_g)) / ipe
        ax.plot(xs, it_g, alpha=0.25, lw=0.5, color="C0")
        ax.plot(xs, it_d, alpha=0.25, lw=0.5, color="C1")
        ep = np.arange(1, len(H[key]["g_loss"]) + 1)
        ax.plot(ep, H[key]["g_loss"], color="C0", lw=2, label="Generator (mean/epoch)")
        ax.plot(ep, H[key]["d_loss"], color="C1", lw=2, label="Discriminator (mean/epoch)")
        ax.set_title(f"{title}: loss G & D (tipis = per-iterasi)")
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    curve(fig.add_subplot(gs[1, 2]), "DDPM (MSE prediksi noise)",
          [("train", H["ddpm"]["train_loss"]), ("val", H["ddpm"]["val_loss"])], logy=True)

    ax = fig.add_subplot(gs[2, :])
    steps = [t for t, _ in traj]
    cols = torch.stack([imgs for _, imgs in traj], dim=0)     # (S, n, 1, 28, 28)
    n = cols.size(1)
    grid = torch.cat([cols[:, i] for i in range(n)], dim=0)   # baris per sampel
    ax.imshow(_row(grid, nrow=len(steps)), cmap="gray")
    ax.set_yticks([])
    ax.set_xticks([(i + 0.5) * 30 for i in range(len(steps))])
    ax.set_xticklabels([f"t={t}" for t in steps], fontsize=8)
    ax.set_title("DDPM reverse diffusion trajectory  x_T -> x_0")
    fig.suptitle("Figur 5 - Training Loss Curves & DDPM Denoising Trajectory", fontsize=14)
    fig.tight_layout()
    return fig, _save(fig, "fig5_losses_ddpm_trajectory.png")


# ---------------------------------------------------------------- Figur pendukung
def figure_gan_diagnostics(H):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for key, ls in (("gan", "-"), ("dcgan", "--")):
        ep = np.arange(1, len(H[key]["d_real"]) + 1)
        axes[0].plot(ep, H[key]["d_real"], ls, label=f"{key}: D(x)")
        axes[0].plot(ep, H[key]["d_fake"], ls, label=f"{key}: D(G(z))")
        axes[1].plot(ep, H[key]["g_grad"], ls, label=key)
    axes[0].set_title("Output Discriminator (rata-rata)")
    axes[1].set_title("Norma gradien Generator  ||dL_G/dtheta_G||")
    for a in axes:
        a.set_xlabel("epoch")
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    fig.suptitle("Figur Pendukung - Diagnostik stabilitas GAN vs DCGAN")
    fig.tight_layout()
    return fig, _save(fig, "figS1_gan_diagnostics.png")


def figure_filters(ae, sae, n=64):
    """Bobot encoder lapisan pertama (784->256) divisualisasikan sebagai filter 28x28."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))
    for ax, (name, W) in zip(axes, (("Vanilla AE", ae.encoder[1].weight), ("Sparse AE", sae.enc1[1].weight))):
        w = W.detach().cpu()[:n].view(-1, 1, 28, 28)
        w = (w - w.amin(dim=(2, 3), keepdim=True)) / (w.amax(dim=(2, 3), keepdim=True) - w.amin(dim=(2, 3), keepdim=True) + 1e-8)
        ax.imshow(_row(w, nrow=8), cmap="gray")
        ax.set_title(f"{name}: 64 filter pertama W1")
        ax.axis("off")
    fig.suptitle("Figur Pendukung - Fitur hidden layer Vanilla AE vs Sparse AE")
    fig.tight_layout()
    return fig, _save(fig, "figS2_ae_vs_sae_filters.png")


def figure_sparsity_hist(h_ae, h_sae):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    bins = np.linspace(0, max(h_ae.max(), h_sae.max()), 60)
    ax.hist(h_ae.flatten(), bins=bins, alpha=0.6, label="Vanilla AE", log=True)
    ax.hist(h_sae.flatten(), bins=bins, alpha=0.6, label="Sparse AE", log=True)
    ax.set_xlabel("nilai aktivasi hidden 256-d (test set)")
    ax.set_ylabel("frekuensi (log)")
    ax.legend()
    ax.set_title("Figur Pendukung - Distribusi aktivasi hidden layer")
    fig.tight_layout()
    return fig, _save(fig, "figS3_activation_hist.png")


def figure_class_hist(cov: dict):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    w = 0.8 / len(cov)
    for i, (k, (ent, hist)) in enumerate(cov.items()):
        ax.bar(np.arange(10) + i * w, hist, width=w, label=f"{k} (H={ent:.3f})")
    ax.set_xticks(np.arange(10) + 0.4 - w / 2)
    ax.set_xticklabels(C.CLASSES, rotation=20, fontsize=8)
    ax.set_ylabel("jumlah sampel")
    ax.legend(fontsize=8)
    ax.set_title("Figur Pendukung - Mode coverage: distribusi kelas sampel sintetis (diklasifikasi MLP)")
    fig.tight_layout()
    return fig, _save(fig, "figS4_mode_coverage.png")

In [ ]:
%%writefile src/experiment.py
"""Orkestrasi eksperimen. Semua state disimpan dalam dict `S` agar mudah dipanggil per-tahap
dari notebook maupun dari run_all.py.
"""
import json
import os

import pandas as pd
import torch

from . import config as C
from . import metrics as M
from . import mlp_limits as L
from . import models as Md
from . import plots as P
from .data import get_loaders, get_splits
from .train import train_gan, train_generic


# ------------------------------------------------------------------ setup
def setup():
    C.set_seed()
    C.make_dirs()
    splits = get_splits()
    S = {"splits": splits, "loaders": get_loaders(splits), "H": {}, "models": {}}
    print(f"Device: {C.DEVICE} | QUICK={C.QUICK}")
    print({k: len(v[0]) for k, v in splits.items()})
    return S


# ------------------------------------------------------------------ Bagian 2 - Tahap 1
@torch.no_grad()
def accuracy(clf, loader):
    clf.eval()
    correct = sum((clf(x.to(C.DEVICE)).argmax(1).cpu() == y).sum().item() for x, y in loader)
    return correct / len(loader.dataset)


def train_classifier(S):
    C.set_seed()
    clf = Md.MLPClassifier()
    S["H"]["mlp"] = train_generic("mlp_classifier", clf, S["loaders"], C.EPOCHS["mlp"], C.LR["mlp"],
                                  loss_fn=lambda m, x, y: m.loss(x, y))
    acc = accuracy(clf, S["loaders"]["test"])
    S["models"]["mlp"] = clf.eval()
    S["mlp_test_acc"] = acc
    print(f"MLP Classifier test accuracy = {acc * 100:.2f}%  (syarat >= 88%: {'OK' if acc >= 0.88 else 'BELUM'})")
    return clf


# ------------------------------------------------------------------ Autoencoder family
def train_autoencoders(S):
    for key, cls in (("ae", Md.VanillaAE), ("sae", Md.SparseAE), ("vae", Md.VAE)):
        C.set_seed()
        m = cls()
        S["H"][key] = train_generic(key, m, S["loaders"], C.EPOCHS[key], C.LR[key])
        S["models"][key] = m.eval()


# ------------------------------------------------------------------ GAN family
def train_gans(S):
    C.set_seed()
    G, D = Md.MLPGenerator(), Md.MLPDiscriminator()
    S["H"]["gan"] = train_gan("gan", G, D, S["loaders"], C.EPOCHS["gan"], C.LR["gan"])
    S["models"]["gan"] = G.eval()

    C.set_seed()
    G, D = Md.DCGenerator(), Md.DCDiscriminator()
    G.apply(Md.dcgan_init)
    D.apply(Md.dcgan_init)
    S["H"]["dcgan"] = train_gan("dcgan", G, D, S["loaders"], C.EPOCHS["dcgan"], C.LR["dcgan"])
    S["models"]["dcgan"] = G.eval()


# ------------------------------------------------------------------ DDPM
def train_ddpm(S):
    C.set_seed()
    m = Md.DDPM()
    S["H"]["ddpm"] = train_generic("ddpm", m, S["loaders"], C.EPOCHS["ddpm"], C.LR["ddpm"], ema=True, amp=True)
    S["models"]["ddpm"] = m.eval()


def sample_fns(S):
    """Fungsi sampling k citra [0,1] dari prior N(0,I) untuk setiap model generatif."""
    Mo = S["models"]
    zg = lambda k: torch.randn(k, C.Z_GAN, device=C.DEVICE)
    return {
        "VAE": lambda k: Mo["vae"].sample(k),
        "Vanilla GAN": lambda k: (Mo["gan"](zg(k)) + 1) / 2,
        "DCGAN": lambda k: (Mo["dcgan"](zg(k)) + 1) / 2,
        "DDPM": lambda k: Mo["ddpm"].sample(k),
    }


# ------------------------------------------------------------------ Bagian 2 - Tahap 2..5
def mlp_limit_experiments(S):
    clf, ae, vae = S["models"]["mlp"], S["models"]["ae"], S["models"]["vae"]
    test_loader = S["loaders"]["test"]
    R = {"test_accuracy": S["mlp_test_acc"]}

    # Tahap 2-3 + diagnostik bobot
    R["weights"] = L.weight_diagnostics(clf, test_loader)
    x8 = _pick_examples(S, 8)
    wt_cls, wt_cls_pre = L.transpose_from_class(clf)
    wt_lat, wt_lat_pre = L.transpose_from_latent(clf, x8)
    pinv_lat = L.pinv_from_latent(clf, x8)
    R["transpose_latent_recon"] = M.recon_metrics(lambda x: L.transpose_from_latent(clf, x)[0], test_loader)
    R["pinv_latent_recon"] = M.recon_metrics(lambda x: L.pinv_from_latent(clf, x), test_loader)

    # Tahap 4: frozen latent decoder
    C.set_seed()
    pipe = L.FrozenPipe(clf, Md.FrozenLatentDecoder())
    S["H"]["frozen_dec"] = train_generic("frozen_decoder", pipe, S["loaders"], C.EPOCHS["frozen_dec"], C.LR["frozen_dec"])
    pipe.eval()
    S["models"]["frozen_pipe"] = pipe
    R["frozen_decoder_recon"] = M.recon_metrics(pipe.reconstruct, test_loader)
    R["vanilla_ae_recon"] = M.recon_metrics(ae.reconstruct, test_loader)

    # Variasi intra-kelas yang tersisa di ruang laten (z_MLP vs z_AE)
    z_mlp, y, mu, sd = L.latent_stats(clf, test_loader)
    z_ae = torch.cat([ae.encode(x.to(C.DEVICE)).cpu() for x, _ in test_loader])
    R["intra_class_var_ratio"] = {"z_MLP": L.intra_class_variance_ratio(z_mlp, y),
                                  "z_AE": L.intra_class_variance_ratio(z_ae, y)}
    R["z_MLP_fraction_zero"] = float((z_mlp == 0).float().mean())

    # Tahap 5: sampling di ruang laten MLP
    g = torch.Generator().manual_seed(C.SEED)
    prior16 = L.prior_sampling(pipe.decoder, mu, sd, 16, generator=g)
    fake = M.generate(lambda k: L.prior_sampling(pipe.decoder, mu, sd, k), C.FID_N)
    real = S["splits"]["test"][0][:C.FID_N]
    R["mlp_latent_sampling_fid"] = M.fid_score(real, fake)
    R["mlp_latent_sampling_coverage"] = M.mode_coverage(clf, fake)[0]
    S["mlp_fake"] = fake

    torch.manual_seed(C.SEED)
    vae16 = S["models"]["vae"].sample(16)
    S["fig3"] = P.figure3_mlp_limits(x8, wt_cls, wt_lat, pinv_lat, pipe.reconstruct(x8.to(C.DEVICE)),
                                     ae.reconstruct(x8.to(C.DEVICE)), prior16, vae16,
                                     wt_class_pre=wt_cls_pre, wt_latent_pre=wt_lat_pre)
    S["mlp_results"] = R
    with open(os.path.join(C.OUT_DIR, "mlp_limits_results.json"), "w") as f:
        json.dump(R, f, indent=2)
    print(json.dumps(R, indent=2))
    return R


def _pick_examples(S, n):
    """Satu contoh per kelas (kelas 0..n-1) dari test set -> grid yang beragam."""
    x, y = S["splits"]["test"]
    idx = [int((y == c).nonzero()[0]) for c in range(n)]
    return x[idx].to(C.DEVICE)


# ------------------------------------------------------------------ Evaluasi kuantitatif (Bagian IV)
@torch.no_grad()
def evaluate_all(S):
    Mo, H, test = S["models"], S["H"], S["loaders"]["test"]
    real = S["splits"]["test"][0][:C.FID_N]
    clf = Mo["mlp"]
    x64 = S["splits"]["test"][0][:C.LATENCY_BATCH].to(C.DEVICE)
    fns = sample_fns(S)
    rows, S["fake"], S["coverage"] = [], {}, {}

    def epoch_time(key):
        return sum(H[key]["epoch_time"]) / len(H[key]["epoch_time"])

    # Autoencoder deterministik + VAE (rekonstruksi)
    for key, name in (("ae", "Vanilla Autoencoder"), ("sae", "Sparse Autoencoder"), ("vae", "Variational Autoencoder (VAE)")):
        m = Mo[key]
        rec = M.recon_metrics(m.reconstruct, test)
        if key == "vae":
            torch.manual_seed(C.SEED)
            lat = M.latency_ms(lambda: fns["VAE"](C.LATENCY_BATCH))
            fake = M.generate(fns["VAE"], C.FID_N)
            fid = M.fid_score(real, fake)
            S["fake"]["VAE"] = fake
            S["coverage"]["VAE"] = M.mode_coverage(clf, fake)
            stab = "Stabil / Konvergen"
            dim = str(C.LATENT_AE)
            cov = S["coverage"]["VAE"][0]
        else:
            lat = M.latency_ms(lambda: m.reconstruct(x64))
            fid, cov = "N/A (Latent Voids)", "N/A"
            stab = "Stabil / Konvergen"
            dim = f"{C.LATENT_AE}" if key == "ae" else f"{C.LATENT_AE} (bottleneck) / {C.SPARSE_HIDDEN} (sparse)"
        rows.append({"Model": name, "Dim Laten": dim, "Train Time/Epoch (s)": epoch_time(key),
                     "Latency (ms/batch 64)": lat[0], "Test MSE": rec["mse"], "Test PSNR (dB)": rec["psnr"],
                     "Test SSIM": rec["ssim"], "FID": fid, "Mode Coverage (H)": cov,
                     "Loss CV (stabilitas)": M.stability_cv(H[key]["iter_loss"]), "Kestabilan Training": stab})

    # GAN, DCGAN, DDPM (generatif murni)
    for key, name, dim in (("gan", "Generative Adversarial Network (GAN)", f"{C.Z_GAN} (prior noise)"),
                           ("dcgan", "Deep Convolutional GAN (DCGAN)", f"{C.Z_GAN} (prior noise)"),
                           ("ddpm", "Denoising Diffusion (DDPM)", "28x28 (dimensi citra)")):
        label = {"gan": "Vanilla GAN", "dcgan": "DCGAN", "ddpm": "DDPM"}[key]
        torch.manual_seed(C.SEED)
        lat = M.latency_ms(lambda: fns[label](C.LATENCY_BATCH), repeats=3 if key == "ddpm" else 10,
                           warmup=1 if key == "ddpm" else 2)
        fake = M.generate(fns[label], C.FID_N)
        S["fake"][label] = fake
        S["coverage"][label] = M.mode_coverage(clf, fake)
        if key == "ddpm":
            cv, stab = M.stability_cv(H[key]["iter_loss"]), "Sangat Stabil"
        else:
            cv = M.stability_cv(H[key]["iter_g"])
            stab = "Rentan osilasi / mode collapse" if key == "gan" else "Cukup stabil (sensitif arsitektur)"
        rows.append({"Model": name, "Dim Laten": dim, "Train Time/Epoch (s)": epoch_time(key),
                     "Latency (ms/batch 64)": lat[0], "Test MSE": "N/A", "Test PSNR (dB)": "N/A",
                     "Test SSIM": "N/A", "FID": M.fid_score(real, fake),
                     "Mode Coverage (H)": S["coverage"][label][0], "Loss CV (stabilitas)": cv,
                     "Kestabilan Training": stab})

    # Referensi: FID antara dua himpunan citra riil (batas bawah praktis untuk N sampel yang sama)
    val_real = S["splits"]["val"][0][:C.FID_N]
    rows.append({"Model": "Referensi: Real (val) vs Real (test)", "Dim Laten": "-", "Train Time/Epoch (s)": "-",
                 "Latency (ms/batch 64)": "-", "Test MSE": "-", "Test PSNR (dB)": "-", "Test SSIM": "-",
                 "FID": M.fid_score(real, val_real), "Mode Coverage (H)": M.mode_coverage(clf, val_real)[0],
                 "Loss CV (stabilitas)": "-", "Kestabilan Training": "-"})

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(C.OUT_DIR, "metrics_table.csv"), index=False)
    with open(os.path.join(C.OUT_DIR, "metrics_table.md"), "w") as f:
        f.write(df.to_markdown(index=False, floatfmt=".4f"))
    with open(os.path.join(C.OUT_DIR, "mode_coverage.json"), "w") as f:
        json.dump(S["coverage"], f, indent=2)
    S["table"] = df
    return df


# ------------------------------------------------------------------ Figur
def _encode_pair(S, model):
    x, y = S["splits"]["test"]
    a, b = C.INTERP_CLASSES
    xa = x[int((y == a).nonzero()[0])][None].to(C.DEVICE)
    xb = x[int((y == b).nonzero()[0])][None].to(C.DEVICE)
    return model.encode(xa), model.encode(xb)


@torch.no_grad()
def _gan_pair(S, G):
    """GAN tidak punya encoder: pilih z_A, z_B yang hasilnya diklasifikasi MLP sebagai dua kelas kontras."""
    clf = S["models"]["mlp"]
    gen = torch.Generator(device="cpu").manual_seed(C.SEED)
    z = torch.randn(2000, C.Z_GAN, generator=gen).to(C.DEVICE)
    prob = torch.softmax(clf((G(z) + 1) / 2), 1)
    a, b = C.INTERP_CLASSES
    return z[prob[:, a].argmax()][None], z[prob[:, b].argmax()][None]


@torch.no_grad()
def make_figures(S):
    Mo, H = S["models"], S["H"]
    x8 = _pick_examples(S, 8)
    S["fig1"] = P.figure1_reconstruction(x8, {"Vanilla AE": Mo["ae"].reconstruct(x8),
                                              "Sparse AE": Mo["sae"].reconstruct(x8),
                                              "VAE": Mo["vae"].reconstruct(x8)})
    S["fig2"] = P.figure2_sampling({
        "A: VAE": S["fake"]["VAE"], "B: Vanilla GAN": S["fake"]["Vanilla GAN"],
        "C: DCGAN": S["fake"]["DCGAN"], "D: DDPM": S["fake"]["DDPM"]})

    alphas = torch.tensor(C.INTERP_ALPHAS, device=C.DEVICE)[:, None]
    interps = {}
    for key, name in (("ae", "Vanilla AE"), ("sae", "Sparse AE"), ("vae", "VAE")):
        za, zb = _encode_pair(S, Mo[key])
        interps[name] = Mo[key].decode((1 - alphas) * za + alphas * zb)
    for key, name in (("gan", "Vanilla GAN"), ("dcgan", "DCGAN")):
        za, zb = _gan_pair(S, Mo[key])
        interps[name] = (Mo[key]((1 - alphas) * za + alphas * zb) + 1) / 2
    S["fig4"] = P.figure4_interpolation(interps, C.INTERP_ALPHAS)

    torch.manual_seed(C.SEED)
    steps = max(1, C.DDPM_T // 10)
    _, traj = Mo["ddpm"].sample(4, traj_every=steps)
    S["fig5"] = P.figure5_losses_and_ddpm(H, traj)

    # Figur pendukung
    P.figure_gan_diagnostics(H)
    P.figure_filters(Mo["ae"], Mo["sae"])
    xt = S["splits"]["test"][0][:2000].to(C.DEVICE)
    h_ae = Mo["ae"].encoder[2](Mo["ae"].encoder[1](xt.flatten(1))).cpu().numpy()
    h_sae = Mo["sae"].hidden(xt).cpu().numpy()
    S["sparsity"] = {"Vanilla AE": float((h_ae < 1e-3).mean()), "Sparse AE": float((h_sae < 1e-3).mean()),
                     "mean|h| Vanilla AE": float(abs(h_ae).sum(1).mean()), "mean|h| Sparse AE": float(abs(h_sae).sum(1).mean())}
    with open(os.path.join(C.OUT_DIR, "sparsity_stats.json"), "w") as f:
        json.dump(S["sparsity"], f, indent=2)
    print("Fraksi aktivasi hidden ~0:", S["sparsity"])
    P.figure_sparsity_hist(h_ae, h_sae)
    P.figure_class_hist(S["coverage"])

In [ ]:
%%writefile src/__init__.py
# package

In [ ]:
%%writefile requirements.txt

torch>=2.1
torchvision>=0.16
torchmetrics[image]>=1.3
torch-fidelity>=0.3
numpy
pandas
tabulate
matplotlib
scipy

In [ ]:
%%writefile run_all.py
"""Menjalankan seluruh pipeline eksperimen secara berurutan.

    python run_all.py              # eksperimen penuh (disarankan GPU)
    QUICK=1 python run_all.py      # smoke test cepat
"""
from src import experiment as E


def main():
    S = E.setup()
    E.train_classifier(S)          # Bagian 2 - Tahap 1
    E.train_autoencoders(S)        # Vanilla AE, Sparse AE, VAE
    E.mlp_limit_experiments(S)     # Bagian 2 - Tahap 2..5 + Figur 3
    E.train_gans(S)                # Vanilla GAN, DCGAN
    E.train_ddpm(S)                # DDPM
    df = E.evaluate_all(S)         # Tabel metrik Bagian IV
    print(df.to_string())
    E.make_figures(S)              # Figur 1, 2, 4, 5 + figur pendukung


if __name__ == "__main__":
    main()

## Setup: seed 42, split 80/10/10 (56.000 / 7.000 / 7.000), batch 128

In [ ]:
import importlib, json
import src.config, src.experiment
importlib.reload(src.config); importlib.reload(src.experiment)
from src import experiment as E, config as C
from IPython.display import Image, display, Markdown
S = E.setup()

## Bagian 2 - Tahap 1: MLP Classifier 784 -> 256 -> 64 -> 10 (Cross-Entropy)
Target akurasi test >= 88%.

In [ ]:
clf = E.train_classifier(S)

## Vanilla AE, Sparse AE, VAE
- Vanilla AE: `784-256-64-256-784`, loss MSE
- Sparse AE: arsitektur sama + penalti `lambda * ||h||_1` pada hidden 256-d
- VAE: `784-256-(mu, logvar in R^64)-256-784`, loss BCE + KL

In [ ]:
E.train_autoencoders(S)

## Bagian 2 - Tahap 2-5: Pembuktian batasan MLP Classifier
Tahap 2 (transpos dari one-hot), Tahap 3 (transpos dari z_MLP, ditambah pseudo-inverse sebagai pembanding),
Tahap 4 (frozen latent decoder vs Vanilla AE), Tahap 5 (sampling z ~ N(mu_z, diag(sigma_z^2))). Menghasilkan **Figur 3**.

In [ ]:
R = E.mlp_limit_experiments(S)
display(Image(S['fig3'][1]))

## Vanilla GAN (MLP) & DCGAN
Adam lr 2e-4, betas (0.5, 0.999); generator memakai non-saturating loss. Dicatat juga D(x), D(G(z)), dan norma gradien G.

In [ ]:
E.train_gans(S)

## DDPM
U-Net kecil dengan time embedding, T = 1000, linear beta 1e-4 -> 0.02, loss MSE prediksi noise, EMA 0.999, mixed precision.

In [ ]:
E.train_ddpm(S)

## Bagian IV - Tabel metrik pada Test Set
MSE, PSNR, dan SSIM untuk model rekonstruksi; FID untuk sampel sintetis dari prior; latency per batch 64; mode coverage; stabilitas (CV loss).

In [ ]:
df = E.evaluate_all(S)
display(df)
print(open(os.path.join(C.OUT_DIR, 'metrics_table.md')).read())

## Bagian V - Lima figur wajib + figur pendukung

In [ ]:
E.make_figures(S)
for f in ['fig1_reconstruction', 'fig2_random_sampling', 'fig3_mlp_classifier_limits',
          'fig4_latent_interpolation', 'fig5_losses_ddpm_trajectory',
          'figS1_gan_diagnostics', 'figS2_ae_vs_sae_filters', 'figS3_activation_hist', 'figS4_mode_coverage']:
    display(Markdown(f'### {f}'))
    display(Image(os.path.join(C.FIG_DIR, f + '.png')))

In [ ]:
print(json.dumps(S['mlp_results'], indent=2))
print('Sparsity:', S['sparsity'])
print('Mode coverage (entropi, histogram kelas):', json.dumps(S['coverage'], indent=1))

## Unduh hasil (figur, tabel, json, dan kode `src/`)

In [ ]:
!cd /content && zip -qr results.zip src run_all.py requirements.txt -x "*/__pycache__/*" && cd "$OUT_DIR" && zip -qr /content/results.zip figures *.csv *.md *.json
from google.colab import files
files.download('/content/results.zip')